In [1]:
from google.cloud import storage, bigquery
import pandas as pd
from pyspark.sql import SparkSession
import datetime
import json

In [2]:
storage_client = storage.Client()
bq_client = bigquery.Client()

In [ ]:
spark = SparkSession.builder.appName("HospitalBToLanding").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/11 09:45:15 INFO SparkEnv: Registering MapOutputTracker
26/05/11 09:45:15 INFO SparkEnv: Registering BlockManagerMaster
26/05/11 09:45:15 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/05/11 09:45:15 INFO SparkEnv: Registering OutputCommitCoordinator


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 35950)
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/socketserver.py", line 317, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 348, in process_request
    self.finish_request(request, client_address)
  File "/opt/conda/lib/python3.11/socketserver.py", line 361, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/opt/conda/lib/python3.11/socketserver.py", line 755, in __init__
    self.handle()
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
                           ^^^^^^
  File "/usr/lib/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates = read_

In [4]:
GCS_BUCKET = "healthcare-bucket-gcp"
HOSPITAL_NAME = "hospital-b"
LANDING_PATH = f"gs://{GCS_BUCKET}/landing/{HOSPITAL_NAME}"
ARCHIVE_PATH = f"gs://{GCS_BUCKET}/landing/{HOSPITAL_NAME}/archive/"
CONFIG_FILE_PATH = f"gs://{GCS_BUCKET}/configs/load_config.csv"

In [5]:
BQ_PROJECT = "healthcare-gcp-495708"
BQ_AUDIT_TABLE = f"{BQ_PROJECT}.temp_dataset.audit_log"
BQ_LOG_TABLE = f"{BQ_PROJECT}.temp_dataset.pipeline_logs"
BQ_TEMP_PATH = f"{GCS_BUCKET}/temp/"

In [6]:
POSTGRES_CONFIG = {
    "url": "jdbc:postgresql://10.98.208.9:5432/hospital_b_db",
    "driver": "org.postgresql.Driver",
    "user": "minhld40",
    "password": "Leducminh1406@"
}

In [7]:
log_entries = []

def log_event(event_type, message, table=None):
    """Log an event and store it in the log list"""
    log_entry = {
        "timestamp": datetime.datetime.now().isoformat(),
        "event_type": event_type,
        "message": message,
        "table": table
    }
    log_entries.append(log_entry)
    print(f"[{log_entry['timestamp']}] {event_type} - {message}")

In [8]:
def save_logs_to_gcs():
    """Save logs to a JSON file and upload to GCS"""
    log_filename = f"pipeline_log_{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}.json"
    log_filepath = f"temp/pipeline_logs/{log_filename}"  
    
    json_data = json.dumps(log_entries, indent=4)

    # Get GCS bucket
    bucket = storage_client.bucket(GCS_BUCKET)
    blob = bucket.blob(log_filepath)
    
    # Upload JSON data as a file
    blob.upload_from_string(json_data, content_type="application/json")

    print(f"Logs successfully saved to GCS at gs://{GCS_BUCKET}/{log_filepath}")

In [9]:
def save_logs_to_bigquery():
    """Save logs to BigQuery"""
    if log_entries:
        log_df = spark.createDataFrame(log_entries)
        log_df.write.format("bigquery") \
            .option("table", BQ_LOG_TABLE) \
            .option("temporaryGcsBucket", BQ_TEMP_PATH) \
            .mode("append") \
            .save()
        print("Logs stored in BigQuery for future analysis")

In [10]:
def move_existing_files_to_archive(table):
    blobs = list(storage_client.bucket(GCS_BUCKET).list_blobs(prefix=f"landing/{HOSPITAL_NAME}/{table}/"))
    existing_files = [blob.name for blob in blobs if blob.name.endswith(".json")]

    if not existing_files:
        log_event("INFO", f"No existing files for table {table}")
        return
    
    run_ts = datetime.datetime.now().strftime("%H%M%S%f")
    for file in existing_files:
        source_blob = storage_client.bucket(GCS_BUCKET).blob(file)

        # Extract Date from File Name
        date_part = file.split("_")[-1].split(".")[0]
        year, month, day = date_part[-4:], date_part[2:4], date_part[:2]
        filename = file.split("/")[-1]
        
        # Move to Archive
        archive_path = f"landing/{HOSPITAL_NAME}/archive/{table}/{year}/{month}/{day}/{run_ts}/{filename}"
        destination_blob = storage_client.bucket(GCS_BUCKET).blob(archive_path)

        # Copy file to archive and delete original
        storage_client.bucket(GCS_BUCKET).copy_blob(source_blob, storage_client.bucket(GCS_BUCKET), destination_blob.name)
        source_blob.delete()

        log_event("INFO", f"Moved {file} to {archive_path}", table=table)

In [11]:
def get_latest_watermark(table_name):
    query = f"""
        SELECT MAX(load_timestamp) AS latest_timestamp
        FROM `{BQ_AUDIT_TABLE}`
        WHERE tablename = '{table_name}' 
          AND data_source = 'hospital_b_db' 
          AND status = 'SUCCESS'
    """
    query_job = bq_client.query(query)
    result = query_job.result()
    for row in result:
        if row.latest_timestamp:
            # Luôn trả về string với format chuẩn
            if isinstance(row.latest_timestamp, datetime.datetime):
                return row.latest_timestamp.strftime("%Y-%m-%d %H:%M:%S")
            return str(row.latest_timestamp)
    return "1900-01-01 00:00:00"

In [12]:
def extract_and_save_to_landing(table, load_type, watermark_col):
    try:
        # Normalize load_type 1 lần
        clean_load_type = load_type.strip().lower()

        last_watermark = get_latest_watermark(table) if clean_load_type == "incremental" else None
        log_event("INFO", f"Latest watermark for {table}: {last_watermark}", table=table)

        query = f"(SELECT * FROM {table}) AS t" if clean_load_type == "full" else \
                f"(SELECT * FROM {table} WHERE {watermark_col} > CAST('{last_watermark}' AS TIMESTAMP)) AS t"

        df = (spark.read.format("jdbc")
                .option("url", POSTGRES_CONFIG["url"])
                .option("user", POSTGRES_CONFIG["user"])
                .option("password", POSTGRES_CONFIG["password"])
                .option("driver", POSTGRES_CONFIG["driver"])
                .option("dbtable", query)
                .load())

        log_event("SUCCESS", f"Successfully extracted data from {table}", table=table)

        df.cache()
        record_count = df.count()

        if record_count == 0:
            log_event("INFO", f"No new records for {table}. Skipping.", table=table)
            df.unpersist()
            return

        move_existing_files_to_archive(table)

        today = datetime.datetime.today().strftime('%d%m%Y')
        JSON_FILE_PATH = f"landing/{HOSPITAL_NAME}/{table}/{table}_{today}.json"

        bucket = storage_client.bucket(GCS_BUCKET)
        blob = bucket.blob(JSON_FILE_PATH)
        blob.upload_from_string(
            df.toPandas().to_json(orient="records", lines=True),
            content_type="application/json"
        )
        log_event("SUCCESS", f"JSON written to gs://{GCS_BUCKET}/{JSON_FILE_PATH}", table=table)

        df.unpersist()

        audit_df = spark.createDataFrame([
            ("hospital_b_db", table, clean_load_type, record_count, datetime.datetime.now(), "SUCCESS")],
            ["data_source", "tablename", "load_type", "record_count", "load_timestamp", "status"])

        (audit_df.write.format("bigquery")
            .option("table", BQ_AUDIT_TABLE)
            .option("temporaryGcsBucket", GCS_BUCKET)
            .mode("append")
            .save())

        log_event("SUCCESS", f"Audit log updated for {table}", table=table)

    except Exception as e:
        log_event("ERROR", f"Error processing {table}: {str(e)}", table=table)

In [13]:
def read_config_file():
    df = spark.read.csv(CONFIG_FILE_PATH, header=True)
    log_event("INFO", "Successfully read the config file")
    return df

In [14]:
config_df = read_config_file()

[2026-05-11T09:45:59.528882] INFO - Successfully read the config file


In [15]:
for row in config_df.collect():
    if row["is_active"] == '1' and row["datasource"] == "hospital_b_db": 
        db, src, table, load_type, watermark, _, targetpath = row
        move_existing_files_to_archive(table)
        extract_and_save_to_landing(table, load_type, watermark)

[2026-05-11T09:46:05.772358] INFO - No existing files for table encounters
[2026-05-11T09:46:06.557066] INFO - Latest watermark for encounters: 1900-01-01 00:00:00
[2026-05-11T09:46:07.038964] SUCCESS - Successfully extracted data from encounters


[2026-05-11T09:46:11.175635] INFO - No existing files for table encounters
[2026-05-11T09:46:12.451067] SUCCESS - JSON written to gs://healthcare-bucket-gcp/landing/hospital-b/encounters/encounters_11052026.json


[2026-05-11T09:46:22.671395] SUCCESS - Audit log updated for encounters
[2026-05-11T09:46:22.707970] INFO - No existing files for table patients
[2026-05-11T09:46:23.385048] INFO - Latest watermark for patients: 1900-01-01 00:00:00
[2026-05-11T09:46:23.460931] SUCCESS - Successfully extracted data from patients
[2026-05-11T09:46:24.197038] INFO - No existing files for table patients
[2026-05-11T09:46:24.823129] SUCCESS - JSON written to gs://healthcare-bucket-gcp/landing/hospital-b/patients/patients_11052026.json


[2026-05-11T09:46:30.833132] SUCCESS - Audit log updated for patients
[2026-05-11T09:46:30.859491] INFO - No existing files for table transactions
[2026-05-11T09:46:31.740787] INFO - Latest watermark for transactions: 1900-01-01 00:00:00
[2026-05-11T09:46:31.827307] SUCCESS - Successfully extracted data from transactions


[2026-05-11T09:46:33.712122] INFO - No existing files for table transactions
[2026-05-11T09:46:34.751578] SUCCESS - JSON written to gs://healthcare-bucket-gcp/landing/hospital-b/transactions/transactions_11052026.json


[2026-05-11T09:46:40.811605] SUCCESS - Audit log updated for transactions
[2026-05-11T09:46:40.834224] INFO - No existing files for table providers
[2026-05-11T09:46:40.834263] INFO - Latest watermark for providers: None
[2026-05-11T09:46:40.894066] SUCCESS - Successfully extracted data from providers
[2026-05-11T09:46:41.391288] INFO - No existing files for table providers
[2026-05-11T09:46:41.762535] SUCCESS - JSON written to gs://healthcare-bucket-gcp/landing/hospital-b/providers/providers_11052026.json


[2026-05-11T09:46:47.851943] SUCCESS - Audit log updated for providers
[2026-05-11T09:46:47.877554] INFO - No existing files for table departments
[2026-05-11T09:46:47.877596] INFO - Latest watermark for departments: None
[2026-05-11T09:46:47.916442] SUCCESS - Successfully extracted data from departments
[2026-05-11T09:46:48.339852] INFO - No existing files for table departments
[2026-05-11T09:46:48.626310] SUCCESS - JSON written to gs://healthcare-bucket-gcp/landing/hospital-b/departments/departments_11052026.json


[2026-05-11T09:46:59.543878] SUCCESS - Audit log updated for departments


In [16]:
save_logs_to_gcs()

Logs successfully saved to GCS at gs://healthcare-bucket-gcp/temp/pipeline_logs/pipeline_log_20260511094710.json


In [ ]:
save_logs_to_bigquery()

Logs stored in BigQuery for future analysis


26/05/11 09:58:29 ERROR YarnClientSchedulerBackend: YARN application has exited unexpectedly with state KILLED! Check the YARN application logs for more details.
26/05/11 09:58:29 ERROR YarnClientSchedulerBackend: Diagnostics message: Application application_1778486467647_0011 was killed by user ducmi at 10.148.0.10
26/05/11 09:58:29 ERROR ApplicationMaster: Exception from Reporter thread.
org.apache.hadoop.yarn.exceptions.ApplicationAttemptNotFoundException: Application attempt appattempt_1778486467647_0011_000001 doesn't exist in ApplicationMasterService cache.
	at org.apache.hadoop.yarn.server.resourcemanager.ApplicationMasterService.allocate(ApplicationMasterService.java:408)
	at org.apache.hadoop.yarn.api.impl.pb.service.ApplicationMasterProtocolPBServiceImpl.allocate(ApplicationMasterProtocolPBServiceImpl.java:60)
	at org.apache.hadoop.yarn.proto.ApplicationMasterProtocol$ApplicationMasterProtocolService$2.callBlockingMethod(ApplicationMasterProtocol.java:106)
	at org.apache.hado

In [ ]:
spark.stop()